# 06 · Tipos de erro, vieses entre grupos e efeito da língua

**Parte A, artigo 1 (exploratória): para que lado o modelo erra.** Superestima × subestima o grau, erro médio com
sinal por grau real e erros de impacto clínico: encaminhamento perdido (falso negativo), doença grave (ICDR 3–4)
classificada como 0–1, encaminhamento desnecessário (falso positivo) e olho normal classificado como referenciável.

**Parte B, artigo 1 (exploratória): vieses entre grupos.** Sexo e faixa etária (duas bases); escolaridade e plano
de saúde (só mBRSET; códigos do PhysioNet: 1 analfabeto … 7 superior completo; 0/1 sem/com plano). Diferenças brutas
podem vir só da mistura de gravidade; por isso a regressão logística (GEE por paciente) é **ajustada pelo grau real e
pelo edema**. Grupos com menos de 50 referenciáveis entram só como descrição.

**Parte C, artigo 2: a língua do prompt muda os tipos de erro?** `main_en` × `main_pt` nas mesmas imagens: taxa de cada
erro, diferença pareada (IC por bootstrap por paciente, McNemar agrupado), transição de graus e erro com sinal por grau.
Depois o 2×2 (instrução × formato) e se o efeito do português é igual entre grupos de escolaridade.

Inválida = encaminhar (como no artigo 1). ICs: bootstrap por paciente, 2.000 reamostras.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import config
pd.set_option("display.max_columns", 50, "display.width", 250, "display.max_rows", 200)
from vlmdr import errors, metrics

In [ ]:
MODELOS = ["medgemma_4b", "medgemma_27b"]   # Gemma 3 e Qwen colapsam numa resposta só: tipos de erro sem sentido
GRUPOS = ["sexo", "faixa_etaria", "escolaridade", "plano_saude"]
d = errors.load(MODELOS)
config.TABLES.mkdir(parents=True, exist_ok=True)
print(d.groupby(["dataset", "model"]).size().unstack())
for v in GRUPOS:
    if v in d:
        print(f"\n{v} (pacientes):", d.drop_duplicates(["dataset", "patient_id"]).groupby("dataset")[v].value_counts().to_dict())

## A1 · Taxas de cada tipo de erro (prompt principal)

In [ ]:
erros = errors.error_table(d)
erros.to_csv(config.TABLES / "erros_tipos.csv", index=False)
erros[[c for c in erros.columns if not c.endswith(" n")]]

## A2 · Erro médio com sinal por grau real (negativo = subestima a doença)

In [ ]:
se = errors.signed_error_by_grade(d)
se.to_csv(config.TABLES / "erros_com_sinal_por_grau.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (ds, g) in zip(axes, se.groupby("dataset")):
    for m, h in g.groupby("model"):
        ax.errorbar(h.icdr + (0.08 if m.endswith("27b") else -0.08), h["erro médio"],
                    yerr=[h["erro médio"] - h.lo, h.hi - h["erro médio"]], marker="o", capsize=3, label=m)
    ax.axhline(0, color="grey", lw=0.8)
    ax.set(title=ds, xlabel="grau ICDR real", xticks=range(5))
axes[0].set_ylabel("erro médio (previsto − real)")
axes[0].legend(frameon=False)
config.FIGURES.mkdir(exist_ok=True)
fig.tight_layout(); fig.savefig(config.FIGURES / "erro_com_sinal_por_grau.png", dpi=200)
se.pivot_table(index=["dataset", "model"], columns="icdr", values="erro médio").round(2)

## B1 · Desempenho e erros por grupo

In [ ]:
sub = []
for ds in ("brset", "mbrset"):
    for m in MODELOS:
        for v in GRUPOS:
            if v in d and d.loc[d.dataset == ds, v].notna().any():
                t = errors.subgroup_table(d, v, m, ds)
                sub.append(t.assign(base=ds, modelo=m, variável=v))
sub = pd.concat(sub, ignore_index=True)
sub = sub[["base", "modelo", "variável"] + [c for c in sub.columns if c not in ("base", "modelo", "variável")]]
sub.to_csv(config.TABLES / "vieses_por_grupo.csv", index=False)
sub

## B2 · Regressão ajustada pelo grau real (GEE, agrupada por paciente)

OR > 1 = o grupo tem mais desse erro do que o grupo de referência (o mais frequente), **com a mesma gravidade**.

In [ ]:
adj = []
for ds in ("brset", "mbrset"):
    for m in MODELOS:
        for v in GRUPOS:
            if v in d and d.loc[d.dataset == ds, v].notna().any():
                for e in ("encaminhamento_perdido", "encaminhamento_desnecessario"):
                    adj.append(errors.adjusted_odds(d, v, m, ds, e))
adj = pd.concat(adj, ignore_index=True)
adj.to_csv(config.TABLES / "vieses_or_ajustada.csv", index=False)
adj.round(3)

## C1 · Inglês × português nas mesmas imagens: muda o tipo de erro?

Usa `main_pt`/`a2` (amostra completa, notebooks 05); se ainda não houver, cai para `main_pt`/`r1` (300 por base).

In [ ]:
w = errors.paired_languages(MODELOS)
print("prompts comparados:", w.attrs["a"], "×", w.attrs["b"], "| imagens:", w.groupby(["dataset", "model"]).size().to_dict())
ling = errors.language_error_shift(w)
ling.to_csv(config.TABLES / "lingua_tipos_de_erro.csv", index=False)
ling

### Transição de graus (linhas = inglês, colunas = português; -1 = sem grau legível)

In [ ]:
for ds in ("brset", "mbrset"):
    for m in MODELOS:
        print(ds, m)
        display(errors.grade_transitions(w, m, ds))

### Erro com sinal por grau real: inglês × português

In [ ]:
def por_lingua(w, lado):
    x = w.rename(columns={f"grade_pred_{lado}": "grade_pred", f"erro_{lado}": "erro"})
    return errors.signed_error_by_grade(x).assign(prompt=w.attrs[lado][0])
se_l = pd.concat([por_lingua(w, "a"), por_lingua(w, "b")])
se_l.to_csv(config.TABLES / "lingua_erro_com_sinal_por_grau.csv", index=False)
fig, axes = plt.subplots(len(MODELOS), 2, figsize=(10, 3.4 * len(MODELOS)), sharey=True, squeeze=False)
for i, m in enumerate(MODELOS):
    for j, ds in enumerate(("brset", "mbrset")):
        ax = axes[i, j]
        for pk, h in se_l[(se_l.model == m) & (se_l.dataset == ds)].groupby("prompt"):
            ax.errorbar(h.icdr + (0.08 if pk.endswith("pt") else -0.08), h["erro médio"],
                        yerr=[h["erro médio"] - h.lo, h.hi - h["erro médio"]], marker="o", capsize=3, label=pk)
        ax.axhline(0, color="grey", lw=0.8)
        ax.set(title=f"{m} · {ds}", xlabel="grau ICDR real", xticks=range(5))
    axes[i, 0].set_ylabel("erro médio (previsto − real)")
axes[0, 0].legend(frameon=False)
fig.tight_layout(); fig.savefig(config.FIGURES / "lingua_erro_com_sinal_por_grau.png", dpi=200)
se_l.pivot_table(index=["model", "dataset", "prompt"], columns="icdr", values="erro médio").round(2)

## C2 · 2×2: o efeito vem da instrução ou do formato da resposta?

Subamostra de robustez, imagens com as quatro células. Efeito principal = média, sobre o outro fator, da mudança
para o português (kappa: em pontos de kappa; erros: em pontos percentuais). Precisa dos notebooks 05.

In [ ]:
cel, efeitos = errors.two_by_two(MODELOS)
if len(cel):
    cel.to_csv(config.TABLES / "lingua_2x2_celulas.csv", index=False)
    efeitos.to_csv(config.TABLES / "lingua_2x2_efeitos.csv", index=False)
    display(cel, efeitos)

## C3 · O efeito do português é igual entre grupos de escolaridade? (mBRSET)

Diferença PT − EN na taxa de encaminhamento perdido e no erro médio dos ICDR ≥ 2, por grupo. Exploratório.

In [ ]:
rows = []
if "escolaridade" in w:
    fa = errors.flags(w.rename(columns=lambda c: c[:-2] if c.endswith("_a") else c))
    fb = errors.flags(w.rename(columns=lambda c: c[:-2] if c.endswith("_b") else c))
    for (m, grp), g in w[(w.dataset == "mbrset") & w.escolaridade.notna()].groupby(["model", "escolaridade"]):
        x = (fb.loc[g.index, "encaminhamento_perdido"] - fa.loc[g.index, "encaminhamento_perdido"]).dropna()
        e = g[g.dr_icdr >= 2]
        est, lo, hi = errors._mean_ci(x.values, g.loc[x.index, "patient_id"].values) if len(x) else (np.nan,) * 3
        rows.append({"modelo": m, "escolaridade": grp, "referenciáveis": len(x),
                     "Δ encaminhamento perdido (pontos %)": metrics.fmt(est, lo, hi) if len(x) else "",
                     "Δ erro médio ICDR>=2": round((e.erro_b - e.erro_a).mean(), 2)})
pd.DataFrame(rows)